In [ ]:
from random import sample

import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv(r'..\data\raw\Sample - Superstore.csv', encoding='cp1252')
print(f"Datasetul este incarcat!\n--------------------------------")
df.info()
print(f'-----------------------\n\nValori nule:\n{df.isna().sum()}')
print(f"\n---------------------\nRanduri, coloane: {df.shape}")
df.columns = df.columns.str.lower().str.replace(' ', '_')
sample_df = df.sample(5)
for i in range(0, len(sample_df.columns), 11):
    print(f"----------Columns {i} to {i+11}------------")
    display(sample_df.iloc[:, i:i+11])


In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'])
df['ship_date'] = pd.to_datetime(df['ship_date'])
delayed_orders = df.loc[df['order_date'] != df['ship_date'], ['order_id', 'order_date', 'ship_date']].copy()

delayed_orders['days_diff'] = (delayed_orders['ship_date'] - delayed_orders['order_date']).dt.days

print(delayed_orders.sort_values(by='days_diff', ascending=False))

In [ ]:
#coloane cu valori unice
#import pandas as pd
#df = pd.read_csv(r'.\Sample - Superstore.csv', encoding='cp1252')
unique_cols = [col for col in df.columns if df[col].nunique() == len(df)]
col_val_uniq = [c for c in df.columns if df[c].nunique() == 1]
print("Coloane care au toate valorile unice:", unique_cols)
print(f'Coloane care au o singura valoare (aceasi) pt toate randurile:', col_val_uniq)

uniqueness_summary = pd.DataFrame({
    'Total Rows': len(df),
    'Unique Count': df.nunique(),
    'Is Entirely Unique': df.nunique() == len(df)
})
print(uniqueness_summary)



top_order_id = df['order_id'].value_counts().index[0]
top_count = df['order_id'].value_counts().iloc[0]

print(f"Top Order ID: {top_order_id} (appears {top_count} times)")

# 2. Select every column for that specific ID
top_order_rows = df[df['order_id'] == top_order_id]

# 3. Sort by another column (like product_id) to see what's unique
display(top_order_rows.sort_values(by=top_order_rows.columns[1]))
df=df.drop(columns='country')
df.info()

In [ ]:
# nu exista mai multe customer_name diferite cu acelasi customer_id

id_name_counts = df.groupby('customer_id')['customer_name'].nunique()


mismatched_ids = id_name_counts[id_name_counts > 1].index


if not mismatched_ids.empty:
    print(f"Urmatoarele {len(mismatched_ids)} IDs au nume muliple:\n")
    mismatches = df[df['customer_id'].isin(mismatched_ids)][['customer_id', 'customer_name']]
    report = mismatches.drop_duplicates().sort_values('customer_id')
    print(report)
else:
    print("Nu exista customer_id care sa aiba mai mut de un singur cutomer_name alocat! Fiecare customer_id are un singur nume.")

In [ ]:
# viceversa: nu exista mai multe customer_ids diferite cu acelasi customer_name

id_id_counts = df.groupby('customer_name')['customer_id'].nunique()


mismatched_ids = id_id_counts[id_id_counts > 1].index

if not mismatched_ids.empty:
    print(f"Urmatoarele {len(mismatched_ids)} IDs au nume muliple:\n")
    mismatches = df[df['customer_name'].isin(mismatched_ids)][['customer_name', 'customer_id']]
    report = mismatches.drop_duplicates().sort_values('customer_name')
    print(report)
else:
    print("Nu exista customer_id care sa aiba mai mut de un singur cutomer_name alocat! Fiecare customer_id are un singur nume.")
    print ("Coloana customer_name este redundanta, in locul ei poate fi utilizata coloana customer_id")
    df=df.drop(columns='customer_name')
df.info()

In [ ]:
# nu exista mai multe product_name diferite cu acelasi product_id

id_name_counts = df.groupby('product_id')['product_name'].nunique()

mismatched_ids = id_name_counts[id_name_counts > 1].index

if not mismatched_ids.empty:
    print(f"Urmatoarele {len(mismatched_ids)} IDs au nume muliple:\n")
    mismatches = df[df['product_id'].isin(mismatched_ids)][['product_id', 'product_name']]
    report = mismatches.drop_duplicates().sort_values('product_id')
    print(report)
    print(f'Exista {len(mismatches)} de nume diferite pentru cele {len(mismatched_ids)} de product_id')
else:
    print(
        "Nu exista product_id care sa aiba mai mut de un singur product_name alocat! Fiecare product_id are un singur nume.")

In [ ]:
# viceversa: nu exista mai multe product_ids diferite cu acelasi product_name

id_id_counts = df.groupby('product_name')['product_id'].nunique()


mismatched_ids = id_id_counts[id_id_counts > 1].index

if not mismatched_ids.empty:
    print(f"Urmatoarele {len(mismatched_ids)} IDs au nume muliple:\n")
    mismatches = df[df['product_name'].isin(mismatched_ids)][['product_name', 'product_id']]
    report = mismatches.drop_duplicates().sort_values('product_name')
    print(f'Exista {len(mismatches)} de id diferite pentru cele {len(mismatched_ids)} de product_name')
    print(report)
else:
    print("Nu exista customer_id care sa aiba mai mut de un singur product_name alocat! Fiecare product_id are un singur nume.")
    #print ("Coloana product_name este redundanta, in locul ei poate fi utilizata coloana product_id")
    #df=df.drop(columns='product_name')

In [ ]:
#customer_id cu mai multe customer orders in acelasi an

df['year'] = pd.to_datetime(df['order_date']).dt.year

order_counts = df.groupby(['customer_id', 'year'])['order_id'].nunique().reset_index()

repeat_customers = order_counts[order_counts['order_id'] > 1]


repeat_customers = repeat_customers.sort_values(by='order_id', ascending=False)

print(repeat_customers)



top_2_repeats =order_counts.sort_values('order_id', ascending=False).head(2)


top_rows = df.merge(top_2_repeats[['customer_id', 'year']], on=['customer_id', 'year'])

display(top_rows.sort_values(by=['customer_id', 'order_date']))

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'])
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month
df['order_day'] = df['order_date'].dt.day
df['ship_date'] = pd.to_datetime(df['ship_date'])
df['ship_year'] = df['ship_date'].dt.year
df['ship_month'] = df['ship_date'].dt.month
df['ship_day'] = df['ship_date'].dt.day

# Calculam intarzierea de livrare pentru a o folosi ca feature
df['order_date'] = pd.to_datetime(df['order_date'])
df['ship_date'] = pd.to_datetime(df['ship_date'])
df['shipping_delay'] = (df['ship_date'] - df['order_date']).dt.days

print(df[['order_date', 'ship_date', 'order_year', 'order_month', 'order_day',  'ship_year', 'ship_month', 'ship_day', 'shipping_delay']].head())


In [ ]:
df['profit_per_unit'] = df['profit'].fillna(0)/df['quantity']
df['sales_per_unit'] = df['sales'].fillna(0)/df['quantity']
df.info()

In [ ]:
#exista o corelatie intre profitul pe customer_id din anul precedent si sales pe customer in anul urmator?
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd


customer_yearly = df.groupby(['customer_id', 'order_year'])[['profit', 'sales']].sum().reset_index()


customer_yearly['order_next_year'] = customer_yearly['order_year'] + 1


analysis_df = pd.merge(
    customer_yearly[['customer_id', 'order_year', 'profit']],
    customer_yearly[['customer_id', 'order_next_year', 'sales']],
    left_on=['customer_id', 'order_year'],
    right_on=['customer_id', 'order_next_year']
)


analysis_df = analysis_df.rename(columns={'sales': 'sales_next'})


if not analysis_df.empty:
    correlation = analysis_df['profit'].corr(analysis_df['sales_next'])
    print(f"Correlation between Previous Year Profit and Following Year Sales: {correlation:.2f}")

    plt.figure(figsize=(10, 6))
    sns.regplot(
        data=analysis_df,
        x='profit',
        y='sales_next',
        scatter_kws={'alpha':0.5, 'color':'teal'},
        line_kws={'color':'red'}
    )

    plt.title(f'Analiză Predictivă: Profit An Precedent vs Vânzări An Viitor (r={correlation:.2f})')
    plt.xlabel('Profit per Client (Anul N)')
    plt.ylabel('Vânzări per Client (Anul N+1)')
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.show()
else:
    print("Nu există suprapuneri între ani pentru a  calcula corelația (ai nevoie de date pe cel puțin 2 ani).")

In [ ]:


correlation_value = analysis_df['profit'].corr(analysis_df['sales_next'])


print(f"{' ANALIZA DE CORELATIE ':=^40}")
print(f"Coeficientul Pearson: {correlation_value:.4f}")


if correlation_value > 0.7:
    interpretare = "Corelație puternică pozitiva (Profitul trecut prezice foarte bine vanzarile)."
elif correlation_value > 0.3:
    interpretare = "Corelație moderată (Exista o tendinta, dar sunt și alți factori implicati)."
elif correlation_value > -0.3:
    interpretare = "Corelație slabă sau inexistentă (Profitul trecut nu influențează direct vânzările)."
else:
    interpretare = "Corelație negativă (Profitul mare ar putea descuraja clienții să revină)."

print(f"Interpretare: {interpretare}")
print(f"{'='*40}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd


customer_strategy = df.groupby(['customer_id', 'order_year']).agg({
    'discount': 'mean',
    'sales': 'sum'
}).reset_index()


customer_strategy['order_next_year'] = customer_strategy['order_year'] + 1


discount_analysis_df = pd.merge(
    customer_strategy[['customer_id', 'order_year', 'discount']],
    customer_strategy[['customer_id', 'order_next_year', 'sales']],
    left_on=['customer_id', 'order_year'],
    right_on=['customer_id', 'order_next_year']
).rename(columns={'sales': 'sales_next', 'discount': 'prev_year_discount'})

# 4. Calculăm Corelația
if not discount_analysis_df.empty:
    corr_discount = discount_analysis_df['prev_year_discount'].corr(discount_analysis_df['sales_next'])

    print(f"{' ANALIZA DISCOUNT VS VANZARI VIITOARE ':=^50}")
    print(f"Corelația Pearson: {corr_discount:.4f}")

    # 5. Vizualizare
    plt.figure(figsize=(10, 6))
    sns.regplot(
        data=discount_analysis_df,
        x='prev_year_discount',
        y='sales_next',
        scatter_kws={'alpha':0.4, 'color':'darkorange'},
        line_kws={'color':'blue'}
    )

    plt.title(f'Impactul Discount-ului (An N) asupra Vânzărilor (An N+1)\nCorelație: {corr_discount:.2f}')
    plt.xlabel('Average Discount oferit (Anul Precedent)')
    plt.ylabel('Total Sales (Anul Următor)')
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.show()

    # Interpretare rapidă
    if corr_discount > 0.3:
        print("REZULTAT: Discount-urile par să stimuleze loialitatea și vânzările viitoare.")
    elif corr_discount < -0.1:
        print("REZULTAT: Atenție! Discount-urile mari corelează cu vânzări mai mici anul viitor.")
    else:
        print("REZULTAT: Nu există o conexiune clară între discount și vânzările viitoare.")
else:
    print("Nu există date suficiente pe mai mulți ani pentru această analiză.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Agregăm datele incluzând și Regiunea
# (Presupunem că un client aparține aceleiași regiuni în ambii ani)
customer_region_strategy = df.groupby(['customer_id', 'order_year', 'region']).agg({
    'discount': 'mean',
    'sales': 'sum'
}).reset_index()

# 2. Creăm variabila pentru anul următor
customer_region_strategy['order_next_year'] = customer_region_strategy['order_year'] + 1

# 3. Self-Merge incluzând regiunea în cheia de join
region_analysis_df = pd.merge(
    customer_region_strategy[['customer_id', 'order_year', 'region', 'discount']],
    customer_region_strategy[['customer_id', 'order_next_year', 'sales']],
    left_on=['customer_id', 'order_year'],
    right_on=['customer_id', 'order_next_year']
).rename(columns={'sales': 'sales_next', 'discount': 'prev_year_discount'})

# 4. Calculăm corelația per regiune pentru print
print(f"{' CORELAȚIE DISCOUNT VS VÂNZĂRI VIITOARE PE REGIUNI ':=^60}")
reg_corrs = region_analysis_df.groupby('region').apply(
    lambda x: x['prev_year_discount'].corr(x['sales_next'])
)
print(reg_corrs)

# 5. Vizualizare cu Seaborn lmplot (Clustering vizual pe regiuni)
g = sns.lmplot(
    data=region_analysis_df,
    x='prev_year_discount',
    y='sales_next',
    col='region',
    hue='region',
    palette='Set2',
    scatter_kws={'alpha': 0.5},
    facet_kws={'sharey': False, 'sharex': True}
)

g.set_axis_labels("Average Discount (An N)", "Total Sales (An N+1)")
(g.fig.suptitle('Impactul Discountului asupra Vânzărilor Viitoare per Regiune', y=1.05, fontsize=16))
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Agregăm datele pe toate dimensiunile cerute
customer_deep_dive = df.groupby(['customer_id', 'order_year', 'region', 'category', 'sub-category']).agg({
    'discount': 'mean',
    'sales': 'sum'
}).reset_index()

# 2. Creăm variabila pentru anul următor (Lag)
customer_deep_dive['order_next_year'] = customer_deep_dive['order_year'] + 1

# 3. Self-Merge (Legăm comportamentul de cumpărare pe aceleași categorii/regiuni)
# Notă: Aceasta verifică dacă discountul la "Chairs" în 2024 a adus vânzări la "Chairs" în 2025 pentru același client
deep_analysis_df = pd.merge(
    customer_deep_dive, 
    customer_deep_dive[['customer_id', 'order_next_year', 'region', 'category', 'sub-category', 'sales']], 
    left_on=['customer_id', 'order_year', 'region', 'category', 'sub-category'], 
    right_on=['customer_id', 'order_next_year', 'region', 'category', 'sub-category'],
    suffixes=('_prev', '_next')
).rename(columns={'discount': 'prev_year_discount'})

# 4. Calculăm corelația pe grupuri (Regiune + Sub-Categorie)
def get_corr(group):
    if len(group) > 5: # Avem nevoie de un eșantion minim pentru relevanță
        return group['prev_year_discount'].corr(group['sales_next'])
    return None

correlation_results = deep_analysis_df.groupby(['region', 'category', 'sub-category']).apply(get_corr).reset_index()
correlation_results.columns = ['region', 'category', 'sub-category', 'correlation']

# Eliminăm valorile nule și sortăm
correlation_results = correlation_results.dropna().sort_values(by='correlation', ascending=False)

# 5. Vizualizare sub formă de Heatmap pentru o Regiune specifică (ex: South)
# Sau un Barplot cu cele mai reactive sub-categorii
plt.figure(figsize=(14, 8))
top_reactive = correlation_results.head(15) # Top 15 combinații unde discountul "funcționează"

sns.barplot(
    data=top_reactive, 
    x='correlation', 
    y='sub-category', 
    hue='region',
    palette='viridis'
)

plt.title('Top 15 Sub-Categorii unde Discountul (An N) generează Vânzări (An N+1)', fontsize=15)
plt.xlabel('Coeficient de Corelație (Eficiența Discountului)')
plt.ylabel('Sub-Categorie')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()

Concluzie: Pentru subcategoria 'Art' discountul pe client are o influenta asupra vaznarilor pe client de la un an la altul pentru regiunea:West

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Redenumim coloana în DataFrame-ul principal (dacă este necesar)
if 'sub_category' in df.columns:
    df = df.rename(columns={'sub_category': 'sub-category'})

# 2. Agregăm Profitul și Vânzările pe toate dimensiunile
customer_profit_dive = df.groupby(['customer_id', 'order_year', 'region', 'category', 'sub-category']).agg({
    'profit': 'sum',
    'sales': 'sum'
}).reset_index()
print(customer_profit_dive)

# 3. Creăm variabila de decalaj (Lag) pentru anul viitor
customer_profit_dive['order_next_year'] = customer_profit_dive['order_year'] + 1

# 4. Self-Merge pentru a lega Profitul trecut de Vânzările viitoare
profit_impact_df = pd.merge(
    customer_profit_dive,
    customer_profit_dive[['customer_id', 'order_next_year', 'region', 'category', 'sub-category', 'sales']],
    left_on=['customer_id', 'order_year', 'region', 'category', 'sub-category'],
    right_on=['customer_id', 'order_next_year', 'region', 'category', 'sub-category'],
    suffixes=('_prev', '_next')
).rename(columns={'profit': 'prev_year_profit', 'sales_next': 'future_sales'})

# 5. Calculăm corelația pe grupuri (Regiune + Sub-category)
def calculate_profit_corr(group):
    if len(group) > 8: # Prag de relevanță pentru eșantion
        return group['prev_year_profit'].corr(group['future_sales'])
    return None

profit_corr_results = profit_impact_df.groupby(['region', 'category', 'sub-category']).apply(
    lambda x: calculate_profit_corr(x), include_groups=False
).reset_index()

profit_corr_results.columns = ['region', 'category', 'sub-category', 'profit_future_sales_corr']

# 6. Vizualizare: Top 15 cele mai predictibile segmente
top_profit_segments = profit_corr_results.dropna().sort_values(by='profit_future_sales_corr', ascending=False).head(15)

plt.figure(figsize=(14, 8))
sns.barplot(
    data=top_profit_segments,
    x='profit_future_sales_corr',
    y='sub-category',
    hue='region',
    palette='magma'
)



plt.title('Top 15 Segmente: Profit (An N) ca Predictor pentru Vânzări (An N+1)', fontsize=15)
plt.xlabel('Coeficient de Corelație (Profit -> Vânzări Viitoare)')
plt.ylabel('Sub-category')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.show()

Concluzie: Pentru subcategoria 'Accesories' profitul pe client are o influenta asupra vanzarilor pe client de la un an la altul pentru regiunea:Est si o influenta medie pentru regiunea: West

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Asigurăm formatul coloanei sub-category
if 'sub_category' in df.columns:
    df = df.rename(columns={'sub_category': 'sub-category'})

# 2. Agregăm Vânzările pe toate dimensiunile (Customer, An, Regiune, Ierarhie Produs)
customer_sales_persistence = df.groupby(['customer_id', 'order_year', 'region', 'category', 'sub-category'])['sales'].sum().reset_index()

# 3. Creăm variabila de decalaj (Lag) pentru anul viitor
customer_sales_persistence['order_next_year'] = customer_sales_persistence['order_year'] + 1

# 4. Self-Merge pentru a lega Vânzările de anul acesta cu cele de anul viitor
sales_lag_df = pd.merge(
    customer_sales_persistence,
    customer_sales_persistence[['customer_id', 'order_next_year', 'region', 'category', 'sub-category', 'sales']],
    left_on=['customer_id', 'order_year', 'region', 'category', 'sub-category'],
    right_on=['customer_id', 'order_next_year', 'region', 'category', 'sub-category'],
    suffixes=('_current', '_next')
).rename(columns={'sales_current': 'sales_prev', 'sales_next': 'sales_future'})

# 5. Calculăm corelația pe grupuri (Regiune + Sub-category)
def calculate_persistence_corr(group):
    if len(group) > 8: # Prag pentru relevanță statistică
        return group['sales_prev'].corr(group['sales_future'])
    return None

sales_corr_results = sales_lag_df.groupby(['region', 'category', 'sub-category']).apply(
    lambda x: calculate_persistence_corr(x), include_groups=False
).reset_index()

sales_corr_results.columns = ['region', 'category', 'sub-category', 'sales_persistence_corr']

# 6. Vizualizare: Top 15 segmente cu cea mai mare retenție a vânzărilor
top_sales_segments = sales_corr_results.dropna().sort_values(by='sales_persistence_corr', ascending=False).head(15)

plt.figure(figsize=(14, 8))
sns.barplot(
    data=top_sales_segments,
    x='sales_persistence_corr',
    y='sub-category',
    hue='region',
    palette='viridis'
)



plt.title('Top 15 Segmente: Persistența Vânzărilor (Cât de predictibil este volumul viitor?)', fontsize=15)
plt.xlabel('Coeficient de Corelație (Sales An N -> Sales An N+1)')
plt.ylabel('Sub-category')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
#verificam daca achizitia unui product_id are vreo influenta asupra vanzarilor pe client de la un an la altul

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Agregăm vânzările pe client, an, produs și regiune
prod_yearly = df.groupby(['customer_id', 'order_year', 'product_id', 'region'])['sales'].sum().reset_index()

# 2. Calculăm vânzările TOTALE ale clientului în anul următor (Target-ul)
total_customer_next_year = df.groupby(['customer_id', 'order_year'])['sales'].sum().reset_index()
total_customer_next_year['order_year_prev'] = total_customer_next_year['order_year'] - 1

# 3. Merge: Legăm produsul cumpărat în Anul N de vânzările totale din Anul N+1
prod_analysis_df = pd.merge(
    prod_yearly,
    total_customer_next_year[['customer_id', 'order_year_prev', 'sales']],
    left_on=['customer_id', 'order_year'],
    right_on=['customer_id', 'order_year_prev'],
    suffixes=('_prod_current', '_total_next')
)

# 4. Funcția de corelație cu prag de relevanță
def calc_prod_corr(group):
    if len(group) > 10: # Cel puțin 10 clienți trebuie să fi cumpărat produsul
        return group['sales_prod_current'].corr(group['sales_total_next'])
    return None

# 5. Aplicăm corelația și rezolvăm eroarea de "Length mismatch"
# Folosim include_groups=False pentru a evita coloane duplicate în rezultat
prod_corrs = prod_analysis_df.groupby(['product_id', 'region']).apply(
    lambda x: calc_prod_corr(x), include_groups=False
).reset_index()

# COREȚIA: Redenumim dinamic ultima coloană (rezultatul corelației)
# Indiferent dacă avem 3, 4 sau 6 coloane, ultima va deveni 'future_value_corr'
prod_corrs.columns = [*prod_corrs.columns[:-1], 'future_value_corr']

# 6. Curățăm datele și selectăm Top 15
prod_corrs = prod_corrs.dropna(subset=['future_value_corr'])
top_anchor_products = prod_corrs.sort_values(by='future_value_corr', ascending=False).head(15)

# 7. Vizualizare
if not top_anchor_products.empty:
    plt.figure(figsize=(12, 8))
    sns.barplot(
        data=top_anchor_products,
        x='future_value_corr',
        y='product_id',
        hue='region',
        palette='magma'
    )
    plt.title('Top 15 Produse Ancoră (Predictori pentru vânzări viitoare)', fontsize=14)
    plt.xlabel('Corelație (Sales Produs An N -> Sales Total Client An N+1)')
    plt.ylabel('Product ID')
    plt.grid(axis='x', linestyle='--', alpha=0.3)
    plt.show()
else:
    print("Nu s-au găsit produse cu destule date (minim 10 apariții) pentru a calcula corelația.")

In [ ]:
from pathlib import Path
df.info()
file_path = Path("transactions_profit_prediction/data/processed/df_preprocessed.csv")

df.to_csv(file_path, index=False)
